In [1]:
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv('gem_api_key')


In [2]:
from langchain.chat_models import init_chat_model
llm =init_chat_model("google_genai:gemini-3-flash-preview",api_key=api_key)
llm.invoke("What is the capital of France?")

ChatGoogleGenerativeAIError: Error calling model 'gemini-3-flash-preview' (PERMISSION_DENIED): 403 PERMISSION_DENIED. {'error': {'code': 403, 'message': 'Your API key was reported as leaked. Please use another API key.', 'status': 'PERMISSION_DENIED'}}

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

from typing import Annotated, TypedDict

class State(TypedDict):
    messages: Annotated[list, add_messages]

def chatbot(state: State) -> State:
    return {"messages": [llm.invoke(state["messages"])]}

builder = StateGraph(State)
builder.add_node("chatbot_node", chatbot)

builder.add_edge(START, "chatbot_node")
builder.add_edge("chatbot_node", END)

graph = builder.compile()

In [ ]:
message = {"role": "user", "content": "Who walked on the moon for the first time? Print only the name"}
# message = {"role": "user", "content": "What is the latest price of MSFT stock?"}
response = graph.invoke({"messages":[message]})

response["messages"]

[HumanMessage(content='Who walked on the moon for the first time? Print only the name', additional_kwargs={}, response_metadata={}, id='490cb365-4786-47a0-9c44-a369c9337573'),
 AIMessage(content=[{'type': 'text', 'text': 'Neil Armstrong', 'extras': {'signature': 'EuACCt0CAQw51sclTXzI6fY4XFyfbQPSfaWF2wATiby9wZXxiU9x255Eqz9hkEuIykExk3+uKMldzWy4mqIIASjmeDok/tHFxz2ujIGYDRH/BHf7Pdn4vDomiVZ7/jGf7WyWpFsquN/pdK1WxF6RQPDfa/sNssgYWmA0Y8Jw5E305I70ZukAT3U0KqZYivy4UqX/cZMk18RInqcQfCWxZ4+tEdZMJ8evxDJlZF84xiFcBCudjRpLFr+B7XNZDweygO6Dk9SkV486SnzQZUQwhjwUFLta/1HAuwjL6Dy7LVerthQMdHx90yBhOS0SO1IJAQhA1RWY+bT+pswLvOBn7EWOUPi1Seo1xL6Tmeczk/6zAkiAVM78J7AeaOEQm7D8BvIk2f0jhI4wPeoEehO0ES/fd2dAXb78oRxmwMr1Uxn4qXhAR5gaSUEMpBGG8G5jrBlsoP7Wb/mdU3RHoWeOGAgG1A=='}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3-flash-preview', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e1a12-a16e-7550-9fe7-178883fb78c3-0', tool_calls=[], invalid_tool_calls=[], 

In [ ]:
state = None
while True:
    in_message = input("You: ")
    if in_message.lower() in {"quit","exit"}:
        break
    if state is None:
        state: State = {
            "messages": [{"role": "user", "content": in_message}]
        }
    else:
        state["messages"].append({"role": "user", "content": in_message})

    state = graph.invoke(state)
    print("Bot:", state["messages"][-1].content)